# Sanity Check: Aritmetica Vettoriale — Ricerca su vocabolari reali

- **EN**: ~21k parole reali estratte dal tokenizer BERT
- **ZH**: ~104k parole reali da **CC-CEDICT** (2-4 caratteri CJK)

Il tokenizer cinese segmenta a singolo carattere → inutile per analogie.
CC-CEDICT dà parole multi-carattere reali (王后, 柏林, 立法者, etc.).

---

## 0. Setup

In [1]:
import gc
import re
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

MODELS_DIR = "../models"
CEDICT_PATH = Path("../data/raw/cedict/cedict_ts.u8")

/Users/gpuzio/Desktop/CODE/THESIS/cls_pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ALL_MODELS = [
    {"label": "E5-large-v2",  "name": "intfloat/e5-large-v2",              "prefix": "query: ",           "lang": "en", "group": "WEIRD"},
    {"label": "BGE-EN-v1.5",  "name": "BAAI/bge-large-en-v1.5",            "prefix": "",                  "lang": "en", "group": "WEIRD"},
    {"label": "Nomic-v1.5",   "name": "nomic-ai/nomic-embed-text-v1.5",    "prefix": "search_document: ", "lang": "en", "group": "WEIRD"},
    {"label": "BGE-ZH-v1.5",  "name": "BAAI/bge-large-zh-v1.5",            "prefix": "",                  "lang": "zh", "group": "Sinic"},
    {"label": "Text2vec-ZH",  "name": "shibing624/text2vec-base-chinese",   "prefix": "",                  "lang": "zh", "group": "Sinic"},
    {"label": "SBERT-ZH-v2",  "name": "DMetaSoul/sbert-chinese-general-v2", "prefix": "",                  "lang": "zh", "group": "Sinic"},
]

In [3]:
# ── Operazioni da testare (solo operandi, nessun atteso) ──────────

EN_ADDITIONS = [
    ("king", "man", "woman"),
    ("Paris", "France", "Germany"),
    ("law", "state", "nature"),
    ("judge", "court", "legislature"),
    ("crime", "punishment", "reward"),
    ("contract", "agreement", "dispute"),
]

ZH_ADDITIONS = [
    ("国王", "男人", "女人"),
    ("巴黎", "法国", "德国"),
    ("法律", "国家", "自然"),
    ("法官", "法院", "立法机关"),
    ("犯罪", "惩罚", "奖励"),
    ("契约", "协议", "纠纷"),
]

EN_SUBTRACTIONS = [
    ("law", "state"),
    ("justice", "power"),
    ("rights", "duty"),
    ("constitution", "democracy"),
    ("crime", "punishment"),
    ("freedom", "order"),
]

ZH_SUBTRACTIONS = [
    ("法律", "国家"),
    ("正义", "权力"),
    ("权利", "义务"),
    ("宪法", "民主"),
    ("犯罪", "惩罚"),
    ("自由", "秩序"),
]

## 1. Funzioni

In [4]:
def free_gpu():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_model(cfg: dict) -> SentenceTransformer:
    free_gpu()
    m = SentenceTransformer(cfg["name"], cache_folder=MODELS_DIR, trust_remote_code=True)
    print(f"  Modello: {cfg['label']} — dim={m.get_sentence_embedding_dimension()}")
    return m


def embed_batch(m, words, prefix, batch_size=256):
    """Embed + L2 normalize, a batch."""
    texts = [f"{prefix}{w}" for w in words] if prefix else list(words)
    vecs = m.encode(texts, convert_to_numpy=True, show_progress_bar=True, batch_size=batch_size)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return vecs / norms


def embed1(m, word, prefix):
    t = f"{prefix}{word}" if prefix else word
    v = m.encode([t], convert_to_numpy=True, show_progress_bar=False)
    n = np.linalg.norm(v[0])
    return v[0] / n if n > 0 else v[0]


def l2n(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

In [5]:
# ── Estrazione vocabolario ────────────────────────────────────────

# EN: dal tokenizer BERT (~21k parole reali)
_RE_EN_WORD = re.compile(r"^[a-zA-Z][a-zA-Z'-]{1,25}$")


def extract_en_words(m: SentenceTransformer) -> list[str]:
    """Estrae parole inglesi reali dal tokenizer BERT."""
    all_tokens = m.tokenizer.get_vocab()
    words = set()
    for token in all_tokens:
        clean = token
        if clean.startswith("##"):
            continue
        if clean.startswith("▁"):
            clean = clean[1:]
        if clean.startswith("Ġ"):
            clean = clean[1:]
        if _RE_EN_WORD.match(clean):
            words.add(clean)
    return sorted(words)


# ZH: da CC-CEDICT (~104k parole, 2-4 caratteri CJK)
_RE_ZH_MULTICHAR = re.compile(r"^[\u4e00-\u9fff]{2,4}$")

# Cache globale: carica una volta sola
_cedict_cache: list[str] | None = None


def load_cedict_words() -> list[str]:
    """
    Carica parole simplified da CC-CEDICT.
    Filtra: solo 2-4 caratteri CJK puri, deduplicate.
    ~104k parole.
    """
    global _cedict_cache
    if _cedict_cache is not None:
        return _cedict_cache

    words = set()
    with open(CEDICT_PATH, encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            # Formato: Traditional Simplified [pinyin] /def/
            parts = line.split(" ", 2)
            if len(parts) >= 2:
                simp = parts[1]
                if _RE_ZH_MULTICHAR.match(simp):
                    words.add(simp)

    _cedict_cache = sorted(words)
    return _cedict_cache


def get_vocab(m: SentenceTransformer, lang: str) -> list[str]:
    """Restituisce il vocabolario appropriato per la lingua."""
    if lang == "zh":
        return load_cedict_words()
    return extract_en_words(m)


def find_neighbors(vec, vocab, vocab_emb, exclude_lower, top_n=15):
    """Top-N vicini, escludendo parole in exclude_lower."""
    sims = cosine_similarity(vec.reshape(1, -1), vocab_emb)[0]
    ranking = sorted(
        [(vocab[i], float(sims[i])) for i in range(len(vocab))
         if vocab[i].lower() not in exclude_lower],
        key=lambda x: x[1], reverse=True,
    )
    return ranking[:top_n]

## 2. Test per un singolo modello

1. Carica il modello
2. Carica vocabolario: **tokenizer** per EN (~21k), **CC-CEDICT** per ZH (~104k)
3. Embedda l'intero vocabolario (ZH: ~2-3 min per modello)
4. Operazioni e ricerca nearest neighbor
5. Scarica il modello

In [6]:
def test_model(cfg: dict) -> dict:
    lang = cfg["lang"]
    prefix = cfg["prefix"]
    additions = ZH_ADDITIONS if lang == "zh" else EN_ADDITIONS
    subtractions = ZH_SUBTRACTIONS if lang == "zh" else EN_SUBTRACTIONS

    m = load_model(cfg)

    # Vocabolario: tokenizer per EN, CC-CEDICT per ZH
    vocab = get_vocab(m, lang)
    source = "CC-CEDICT" if lang == "zh" else "tokenizer"
    print(f"  Vocabolario ({source}): {len(vocab)} parole")
    if lang == "zh":
        print(f"    campione: {', '.join(vocab[::len(vocab)//8][:8])}")

    # Embed intero vocabolario
    print(f"  Embedding di {len(vocab)} parole...")
    vocab_emb = embed_batch(m, vocab, prefix)
    print(f"  Shape: {vocab_emb.shape}")

    # A - B + C = ?
    add_results = []
    for a, b, c in additions:
        va, vb, vc = embed1(m, a, prefix), embed1(m, b, prefix), embed1(m, c, prefix)
        res = l2n(va - vb + vc)
        exclude = {a.lower(), b.lower(), c.lower()}
        top = find_neighbors(res, vocab, vocab_emb, exclude)
        add_results.append({"op": f"{a} - {b} + {c}", "top": top})

    # A - B = ?
    sub_results = []
    for a, b in subtractions:
        va, vb = embed1(m, a, prefix), embed1(m, b, prefix)
        raw = va - vb
        res = l2n(raw)
        exclude = {a.lower(), b.lower()}
        top = find_neighbors(res, vocab, vocab_emb, exclude)
        sub_results.append({
            "op": f"{a} - {b}",
            "top": top,
            "cos_ab": float(va @ vb),
            "norm_r": float(np.linalg.norm(raw)),
            "max_sim": top[0][1] if top else 0,
        })

    dim = vocab_emb.shape[1]
    n_vocab = len(vocab)

    del m, vocab_emb
    free_gpu()

    return {
        "label": cfg["label"], "group": cfg["group"], "lang": lang,
        "dim": dim, "n_vocab": n_vocab, "vocab_source": source,
        "vocab_words": vocab,
        "additions": add_results, "subtractions": sub_results,
    }

## 3. Esecuzione su tutti e 6 i modelli

- EN: ~30s per modello (21k parole dal tokenizer)
- ZH: ~2-3 min per modello (104k parole da CC-CEDICT)

In [7]:
all_results = {}

for i, cfg in enumerate(ALL_MODELS):
    print(f"\n{'=' * 70}")
    print(f"  [{i+1}/{len(ALL_MODELS)}] {cfg['group']} — {cfg['label']}")
    print(f"{'=' * 70}")

    res = test_model(cfg)
    all_results[cfg["label"]] = res

    print(f"\n  Vocab: {res['n_vocab']} parole, dim={res['dim']}")
    for r in res["additions"]:
        top1 = r["top"][0][0] if r["top"] else "???"
        print(f"    {r['op']:<30s} → {top1}")

print(f"\n{'=' * 70}")
print("  Fatto.")
print(f"{'=' * 70}")


  [1/6] WEIRD — E5-large-v2


  Modello: E5-large-v2 — dim=1024
  Vocabolario (tokenizer): 21719 parole
  Embedding di 21719 parole...


Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Batches:   1%|          | 1/85 [00:01<02:13,  1.59s/it]

Batches:   2%|▏         | 2/85 [00:01<01:10,  1.17it/s]

...

Batches:  99%|█████████▉| 402/405 [00:41<00:00, 11.14it/s]

Batches: 100%|█████████▉| 404/405 [00:41<00:00, 11.15it/s]

Batches: 100%|██████████| 405/405 [00:41<00:00,  9.82it/s]

  Shape: (103439, 768)



  Vocab: 103439 parole, dim=768
    国王 - 男人 + 女人                   → 女皇
    巴黎 - 法国 + 德国                   → 德国人
    法律 - 国家 + 自然                   → 自然而然
    法官 - 法院 + 立法机关                 → 行政机关
    犯罪 - 惩罚 + 奖励                   → 奖品
    契约 - 协议 + 纠纷                   → 争讼

  Fatto.


## 4. Vocabolario di ricerca per modello

In [8]:
print("Vocabolario di ricerca per modello:")
print("=" * 60)
for label, res in all_results.items():
    src = res.get("vocab_source", "tokenizer")
    print(f"  {label:<18s}  {res['n_vocab']:>6d} parole  (dim={res['dim']}, fonte={src})")
    sample = res["vocab_words"][::len(res["vocab_words"])//10][:10]
    print(f"    campione: {', '.join(sample)}")

Vocabolario di ricerca per modello:
  E5-large-v2          21719 parole  (dim=1024, fonte=tokenizer)
    campione: aa, blooms, contraction, engineers, hades, labs, nedra, prompting, sessions, theater
  BGE-EN-v1.5          21719 parole  (dim=1024, fonte=tokenizer)
    campione: aa, blooms, contraction, engineers, hades, labs, nedra, prompting, sessions, theater
  Nomic-v1.5           21719 parole  (dim=768, fonte=tokenizer)
    campione: aa, blooms, contraction, engineers, hades, labs, nedra, prompting, sessions, theater
  BGE-ZH-v1.5         103439 parole  (dim=1024, fonte=CC-CEDICT)
    campione: 一一, 元长乡, 合字, 安抵, 我的世界, 朝鲜半岛, 滨湖区, 秦镜高悬, 落腮胡子, 邪僻
  Text2vec-ZH         103439 parole  (dim=768, fonte=CC-CEDICT)
    campione: 一一, 元长乡, 合字, 安抵, 我的世界, 朝鲜半岛, 滨湖区, 秦镜高悬, 落腮胡子, 邪僻
  SBERT-ZH-v2         103439 parole  (dim=768, fonte=CC-CEDICT)
    campione: 一一, 元长乡, 合字, 安抵, 我的世界, 朝鲜半岛, 滨湖区, 秦镜高悬, 落腮胡子, 邪僻


## 5. Risultati completi per modello

- EN: top-15 cercati tra ~21k parole (tokenizer)
- ZH: top-15 cercati tra ~104k parole (CC-CEDICT)

In [9]:
for label, res in all_results.items():
    print(f"\n{'=' * 70}")
    print(f"  {res['group']} — {label}  (dim={res['dim']}, vocab={res['n_vocab']})")
    print(f"{'=' * 70}")

    print(f"\n  A - B + C = ?   (cercato tra {res['n_vocab']} parole)")
    print(f"  {'─' * 60}")
    for r in res["additions"]:
        print(f"\n  {r['op']} = ?")
        for i, (w, s) in enumerate(r["top"]):
            print(f"    #{i+1:2d}  {w:<20s}  sim = {s:.4f}")

    print(f"\n  A - B = ?   (cercato tra {res['n_vocab']} parole)")
    print(f"  {'─' * 60}")
    for r in res["subtractions"]:
        q = "BUONO" if r["max_sim"] > 0.15 else "DEBOLE" if r["max_sim"] > 0.05 else "PROBLEMATICO"
        print(f"\n  {r['op']} = ?")
        print(f"    cos(A,B)={r['cos_ab']:.4f}  ‖A-B‖={r['norm_r']:.4f}  max_sim={r['max_sim']:.4f} [{q}]")
        for i, (w, s) in enumerate(r["top"]):
            print(f"    #{i+1:2d}  {w:<20s}  sim = {s:.4f}")


  WEIRD — E5-large-v2  (dim=1024, vocab=21719)

  A - B + C = ?   (cercato tra 21719 parole)
  ────────────────────────────────────────────────────────────

  king - man + woman = ?
    # 1  queen                 sim = 0.9050
    # 2  women                 sim = 0.8720
    # 3  kings                 sim = 0.8646
    # 4  wife                  sim = 0.8618
    # 5  princess              sim = 0.8561
    # 6  monarch               sim = 0.8537
    # 7  empress               sim = 0.8524
    # 8  mommy                 sim = 0.8464
    # 9  she                   sim = 0.8460
    #10  daughter              sim = 0.8455
    #11  bride                 sim = 0.8424
    #12  royal                 sim = 0.8395
    #13  aunt                  sim = 0.8389
    #14  sister                sim = 0.8388
    #15  kate                  sim = 0.8383

  Paris - France + Germany = ?
    # 1  berlin                sim = 0.9160
    # 2  munich                sim = 0.9122
    # 3  german                sim = 

## 6. Tabella: A - B + C — top-1 per modello

In [10]:
en_ops = [f"{a}-{b}+{c}" for a, b, c in EN_ADDITIONS]
zh_ops = [f"{a}-{b}+{c}" for a, b, c in ZH_ADDITIONS]

print("TOP-1 per A - B + C (ricerca sull'intero tokenizer)")
print("=" * 110)

for group_lang, ops in [("WEIRD (EN)", en_ops), ("Sinic (ZH)", zh_ops)]:
    lang = "en" if "EN" in group_lang else "zh"
    print(f"\n{'':18s}", end="")
    for op in ops:
        print(f"  {op[:16]:>16s}", end="")
    print()
    print("-" * 110)
    for label, res in all_results.items():
        if res["lang"] != lang: continue
        print(f"{label:<18s}", end="")
        for r in res["additions"]:
            top1 = r["top"][0][0] if r["top"] else "???"
            print(f"  {top1:>16s}", end="")
        print()

TOP-1 per A - B + C (ricerca sull'intero tokenizer)

                      king-man+woman  Paris-France+Ger  law-state+nature  judge-court+legi  crime-punishment  contract-agreeme
--------------------------------------------------------------------------------------------------------------
E5-large-v2                    queen            berlin              laws        legislator           rewards          disputes
BGE-EN-v1.5                   female            berlin           science        legislator           rewards          disputes
Nomic-v1.5                     kings            berlin              laws        legislator            crimes          disputes

                            国王-男人+女人          巴黎-法国+德国          法律-国家+自然        法官-法院+立法机关          犯罪-惩罚+奖励          契约-协议+纠纷
--------------------------------------------------------------------------------------------------------------
BGE-ZH-v1.5                       女王                柏林               自然法              立法委

## 7. Tabella: A - B — top-3 per modello

In [11]:
en_sops = [f"{a}-{b}" for a, b in EN_SUBTRACTIONS]
zh_sops = [f"{a}-{b}" for a, b in ZH_SUBTRACTIONS]

print("TOP-3 per A - B (ricerca sull'intero tokenizer)")
print("=" * 100)

for group_lang, sops in [("WEIRD (EN)", en_sops), ("Sinic (ZH)", zh_sops)]:
    lang = "en" if "EN" in group_lang else "zh"
    group_results = {l: r for l, r in all_results.items() if r["lang"] == lang}

    for j, sop in enumerate(sops):
        print(f"\n  {sop}")
        print(f"  {'─' * 90}")
        for label, res in group_results.items():
            top3 = res["subtractions"][j]["top"][:3]
            t3_str = ", ".join(f"{w}({s:.3f})" for w, s in top3)
            ms = res["subtractions"][j]["max_sim"]
            print(f"    {label:<16s}  max={ms:.4f}  → {t3_str}")

TOP-3 per A - B (ricerca sull'intero tokenizer)

  law-state
  ──────────────────────────────────────────────────────────────────────────────────────────
    E5-large-v2       max=0.1852  → lawyer(0.185), laws(0.184), lawyers(0.166)
    BGE-EN-v1.5       max=0.2541  → jurisprudence(0.254), laws(0.233), medicine(0.218)
    Nomic-v1.5        max=0.3347  → laws(0.335), lawyers(0.257), lawyer(0.232)

  justice-power
  ──────────────────────────────────────────────────────────────────────────────────────────
    E5-large-v2       max=0.1855  → justices(0.186), judiciary(0.185), judicial(0.174)
    BGE-EN-v1.5       max=0.3340  → injustice(0.334), sentencing(0.254), jurist(0.252)
    Nomic-v1.5        max=0.2371  → injustice(0.237), justices(0.230), judge(0.203)

  rights-duty
  ──────────────────────────────────────────────────────────────────────────────────────────
    E5-large-v2       max=0.1503  → records(0.150), freedoms(0.146), championships(0.146)
    BGE-EN-v1.5       max=0.2057  →

## 8. Consistenza intra-gruppo

In [12]:
print("CONSISTENZA — top-1 uguale tra i 3 modelli dello stesso gruppo?")
print("=" * 80)

for gname, ops_a, ops_s in [
    ("WEIRD", EN_ADDITIONS, EN_SUBTRACTIONS),
    ("Sinic", ZH_ADDITIONS, ZH_SUBTRACTIONS),
]:
    gres = {l: r for l, r in all_results.items() if r["group"] == gname}
    mls = list(gres.keys())

    print(f"\n  {gname}:")

    for section, ops, key in [("A-B+C", ops_a, "additions"), ("A-B", ops_s, "subtractions")]:
        print(f"\n  {section}:")
        for j in range(len(ops)):
            if len(ops[j]) == 3:
                a, b, c = ops[j]
                opstr = f"{a}-{b}+{c}"
            else:
                a, b = ops[j]
                opstr = f"{a}-{b}"

            top1s = []
            for ml in mls:
                t = gres[ml][key][j]["top"]
                top1s.append(t[0][0] if t else "???")

            unique = len(set(t.lower() for t in top1s))
            tag = "CONSISTENTE" if unique == 1 else f"DIVERGE ({unique})"
            entries = " | ".join(f"{ml}: {t}" for ml, t in zip(mls, top1s))
            print(f"    {opstr:<24s} {entries}  → {tag}")

CONSISTENZA — top-1 uguale tra i 3 modelli dello stesso gruppo?

  WEIRD:

  A-B+C:
    king-man+woman           E5-large-v2: queen | BGE-EN-v1.5: female | Nomic-v1.5: kings  → DIVERGE (3)
    Paris-France+Germany     E5-large-v2: berlin | BGE-EN-v1.5: berlin | Nomic-v1.5: berlin  → CONSISTENTE
    law-state+nature         E5-large-v2: laws | BGE-EN-v1.5: science | Nomic-v1.5: laws  → DIVERGE (2)
    judge-court+legislature  E5-large-v2: legislator | BGE-EN-v1.5: legislator | Nomic-v1.5: legislator  → CONSISTENTE
    crime-punishment+reward  E5-large-v2: rewards | BGE-EN-v1.5: rewards | Nomic-v1.5: crimes  → DIVERGE (2)
    contract-agreement+dispute E5-large-v2: disputes | BGE-EN-v1.5: disputes | Nomic-v1.5: disputes  → CONSISTENTE

  A-B:
    law-state                E5-large-v2: lawyer | BGE-EN-v1.5: jurisprudence | Nomic-v1.5: laws  → DIVERGE (3)
    justice-power            E5-large-v2: justices | BGE-EN-v1.5: injustice | Nomic-v1.5: injustice  → DIVERGE (2)
    rights-duty       

## 9. Confronto cross-linguistico

Per ogni decomposizione, top-5 di tutti e 6 i modelli.

In [13]:
for (en_a, en_b), (zh_a, zh_b) in zip(EN_SUBTRACTIONS, ZH_SUBTRACTIONS):
    print(f"\n{'=' * 90}")
    print(f"  EN: {en_a} - {en_b}    |    ZH: {zh_a} - {zh_b}")
    print(f"{'=' * 90}")

    en_idx = next(i for i, (a, b) in enumerate(EN_SUBTRACTIONS) if a == en_a and b == en_b)
    zh_idx = next(i for i, (a, b) in enumerate(ZH_SUBTRACTIONS) if a == zh_a and b == zh_b)

    for label, res in all_results.items():
        idx = en_idx if res["lang"] == "en" else zh_idx
        dec = res["subtractions"][idx]
        top5_str = ", ".join(f"{w}({s:.3f})" for w, s in dec["top"][:5])
        print(f"  [{res['group']:5s}] {label:<16s}  → {top5_str}")


  EN: law - state    |    ZH: 法律 - 国家
  [WEIRD] E5-large-v2       → lawyer(0.185), laws(0.184), lawyers(0.166), legal(0.137), barrister(0.124)
  [WEIRD] BGE-EN-v1.5       → jurisprudence(0.254), laws(0.233), medicine(0.218), finance(0.207), lawyer(0.207)
  [WEIRD] Nomic-v1.5        → laws(0.335), lawyers(0.257), lawyer(0.232), legislation(0.231), legal(0.214)
  [Sinic] BGE-ZH-v1.5       → 刑律(0.310), 律法(0.309), 法律责任(0.305), 法理(0.282), 法律制裁(0.277)
  [Sinic] Text2vec-ZH       → 律法(0.494), 法理(0.399), 摩西律法(0.375), 律条(0.373), 司法(0.359)
  [Sinic] SBERT-ZH-v2       → 法令(0.193), 法度(0.191), 判例法(0.190), 定律(0.185), 法定(0.184)

  EN: justice - power    |    ZH: 正义 - 权力
  [WEIRD] E5-large-v2       → justices(0.186), judiciary(0.185), judicial(0.174), injustice(0.163), courtroom(0.149)
  [WEIRD] BGE-EN-v1.5       → injustice(0.334), sentencing(0.254), jurist(0.252), fairness(0.249), sentenced(0.249)
  [WEIRD] Nomic-v1.5        → injustice(0.237), justices(0.230), judge(0.203), jurist(0.203), judicial

## 10. Diagnostica: max similarity

In [14]:
print("max_sim del residuo A-B (cercato tra l'intero vocabolario del tokenizer)")
print("(> 0.15 OK, 0.05-0.15 ~, < 0.05 !!)")
print("=" * 100)

for group_lang, sops in [("WEIRD", en_sops), ("Sinic", zh_sops)]:
    lang = "en" if group_lang == "WEIRD" else "zh"
    print(f"\n{'':18s}", end="")
    for op in sops:
        print(f"  {op:>14s}", end="")
    print()
    print("-" * 100)
    for label, res in all_results.items():
        if res["lang"] != lang: continue
        print(f"{label:<18s}", end="")
        for r in res["subtractions"]:
            ms = r["max_sim"]
            tag = "OK" if ms > 0.15 else ("~ " if ms > 0.05 else "!!")
            print(f"  {ms:>10.4f} {tag}", end="")
        print()

max_sim del residuo A-B (cercato tra l'intero vocabolario del tokenizer)
(> 0.15 OK, 0.05-0.15 ~, < 0.05 !!)

                         law-state   justice-power     rights-duty  constitution-democracy  crime-punishment   freedom-order
----------------------------------------------------------------------------------------------------
E5-large-v2             0.1852 OK      0.1855 OK      0.1503 OK      0.1661 OK      0.2041 OK      0.2663 OK
BGE-EN-v1.5             0.2541 OK      0.3340 OK      0.2057 OK      0.2091 OK      0.3010 OK      0.3511 OK
Nomic-v1.5              0.3347 OK      0.2371 OK      0.1829 OK      0.2726 OK      0.2898 OK      0.3375 OK

                             法律-国家           正义-权力           权利-义务           宪法-民主           犯罪-惩罚           自由-秩序
----------------------------------------------------------------------------------------------------
BGE-ZH-v1.5             0.3103 OK      0.3221 OK      0.3825 OK      0.3398 OK      0.4093 OK      0.3850 OK
Text2vec-ZH

## 11. Conclusioni

Risultati cercati tra vocabolari reali:
- EN: ~21k parole dal tokenizer BERT
- ZH: ~104k parole da CC-CEDICT (2-4 caratteri)

Domande:

1. `国王 - 男人 + 女人` → esce 王后 (regina) o qualcosa di sensato?
2. Le analogie giuridiche ZH producono parole multi-carattere interpretabili?
3. Le decomposizioni A - B hanno top-5 semanticamente coerenti?
4. max_sim > 0.05 ovunque? Se no, il residuo è nel vuoto.
5. I 3 modelli dello stesso gruppo sono consistenti?
6. EN e ZH sono confrontabili adesso che ZH ha parole vere?